# 01 · Data retrieval, cleaning & validation

**Charting Boulder — BVSD enrollment-demography pipeline, stage 1 of 2.**
This notebook acquires every source pinned in `data/raw/README.md`, cleans each into a tidy
shape, **validates** it, and writes the `data/processed/` contract that
`02-boulder-enrollment-demography.ipynb` consumes. Modeling, projection, and visualization
live entirely in notebook 02; this notebook does no analysis.

**Two switches.** `LIVE = True` attempts real downloads. `ALLOW_FALLBACK = True` lets any source
that is key-gated, unreachable, or absent fall back to a **schema-correct synthetic** stand-in so
the downstream notebook always runs. Every output records its `mode` (`live`/`synthetic`) and a
sha256 in `data/processed/manifest.json`, so it is always unambiguous which numbers are real.

| Source | Real retrieval here? | Why |
|---|---|---|
| similar-boulder.json (registry) | ✅ live (GitHub raw) | small public file |
| IRS county→county flows | ✅ live (GitHub raw) | public flat file, 1990–2010 |
| DOLA single-year-of-age (CO) | ✅ live (googleapis CSV) | public; real CO county ages 1990–2060 |
| Zillow ZHVI (prices) | ✅ live (filtered) | public direct CSV |
| NHGIS place/county SYOA | ⚙ scaffold + fallback | needs IPUMS API key + extract latency |
| Hauer county control (SSP2) | ⚙ scaffold + fallback | OSF bulk file; large |
| Census BPS permits | ⚙ scaffold + fallback | fiddly multi-file parse |
| National Zoning Atlas | ⚙ scaffold + fallback | data-access gated |

## 1 · Setup, fetch cache, and manifest

Idempotent downloader (hash + on-disk cache in `data/raw/`), a provenance manifest, and the
project frame (single-year ages 0–85, decennial base 1990–2020, 10-year steps to 2050).

In [1]:
import io, os, json, hashlib, time, warnings
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.options.display.max_columns = 100

SEED = 20260611
RNG = np.random.default_rng(SEED)

LIVE = True              # attempt real downloads
ALLOW_FALLBACK = True    # synthetic stand-in when a source is gated/unreachable
UA = {"User-Agent": "Mozilla/5.0 (charting-boulder research pipeline)"}

RAW = Path("data/raw"); PROC = Path("data/processed")
for p in (RAW, PROC):
    p.mkdir(parents=True, exist_ok=True)

# frame (must match notebook 02)
BASE_YEARS = [1990, 2000, 2010, 2020]
PROJ_YEARS = [2030, 2040, 2050]
MAX_AGE = 85
AGES = np.arange(0, MAX_AGE + 1)

MANIFEST = []
def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:16]

def record(name, path, mode, source_url, rows):
    MANIFEST.append({"name": name, "path": str(path), "mode": mode,
                     "source_url": source_url, "rows": int(rows),
                     "sha256_16": _sha256(path),
                     "retrieved_at": datetime.now(timezone.utc).isoformat(timespec="seconds")})
    print(f"  [{mode:9s}] {name:26s} rows={rows:<8d} -> {path}")

print("retrieval setup ok · LIVE =", LIVE, "· fallback =", ALLOW_FALLBACK)

retrieval setup ok · LIVE = True · fallback = True


In [2]:
import requests
def fetch(url, fname, timeout=90, binary=True):
    # idempotent: cache to data/raw/fname; return local path. Raises on failure.
    dest = RAW / fname
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    r = requests.get(url, headers=UA, timeout=timeout, stream=True)
    r.raise_for_status()
    with open(dest, "wb") as f:
        for chunk in r.iter_content(chunk_size=1 << 20):
            if chunk:
                f.write(chunk)
    if dest.stat().st_size == 0:
        raise IOError("empty download")
    return dest

def cap_ages_85(df_age, age_col="age", pop_col="pop", keys=("fips", "year")):
    # collapse single-year ages >=85 into an 85+ open interval; keep 0..85
    d = df_age.copy()
    d[age_col] = d[age_col].clip(upper=MAX_AGE)
    return d.groupby(list(keys) + [age_col], as_index=False)[pop_col].sum()

def synth_age_long(fips_list, years, id_name, seed_offset):
    # schema-correct synthetic single-year age, tidy long. NEUTRAL (no rigging).
    rng = np.random.default_rng(SEED + seed_offset)
    base = np.exp(-0.0008 * (AGES - 38) ** 2) + 0.15; base /= base.sum()
    recs = []
    for f in fips_list:
        size = rng.uniform(8_000, 130_000); tilt = rng.uniform(0.7, 1.3)
        prof = base.copy(); prof[:18] *= tilt; prof /= prof.sum()
        drift = rng.normal(0, 0.04, len(years)).cumsum()
        for j, y in enumerate(years):
            v = prof.copy(); v[:18] *= (1 + drift[j]); v = np.clip(v, 1e-6, None)
            v = v / v.sum() * size * rng.uniform(0.97, 1.03)
            for a in AGES:
                recs.append((f, y, int(a), float(v[a])))
    return pd.DataFrame(recs, columns=[id_name, "year", "age", "pop"])
print("fetch + helpers ready")

fetch + helpers ready


## 2 · Place registry  ·  *live: similar-boulder.json*

Tier 2 (college towns) comes from the user's published basket; Tier 1 (City of Boulder +
Boulder County ring) is defined here. `county_fips` is stored as a JSON list to carry Erie's
Boulder+Weld straddle.

In [4]:
RING_T1 = [  # name, place_fips, county_fips list, tier
    ("Boulder city, CO",    "0807850", ["08013"],          1),
    ("Longmont city, CO",   "0845970", ["08013"],          1),
    ("Lafayette city, CO",  "0842495", ["08013"],          1),
    ("Louisville city, CO", "0846465", ["08013"],          1),
    ("Superior town, CO",   "0875070", ["08013"],          1),
    ("Erie town, CO",       "0825625", ["08013", "08123"], 1),
]
reg_mode = "synthetic"; src = "hardcoded"
rows = []
if LIVE:
    try:
        p = fetch("https://github.com/brianckeegan/charting-boulder/blob/main/2025-06-population/similar-boulder.json")
        js = json.loads(Path(p).read_text())
        for e in js:
            nm = f"{e['city']} ({e['state']})"
            rows.append((nm, str(e["place_fips"]).zfill(7), [str(e["county_fips"]).zfill(5)], 2))
        reg_mode = "live"; src = "github:brianckeegan/charting-boulder"
    except Exception as ex:
        print("  registry live failed:", ex)
        if not ALLOW_FALLBACK: raise
# Tier-1 ring + dedup (drop any Boulder dup coming from the json)
seen = set(); merged = []
for nm, pf, cf, t in RING_T1 + rows:
    if pf in seen: continue
    seen.add(pf); merged.append((nm, pf, json.dumps(cf), t))
places_registry = pd.DataFrame(merged, columns=["name", "place_fips", "county_fips_json", "tier"])
places_registry.to_csv(PROC / "places_registry.csv", index=False)
record("places_registry", PROC / "places_registry.csv", reg_mode, src, len(places_registry))
# validate
assert places_registry["place_fips"].str.len().eq(7).all()
assert places_registry["tier"].isin([1, 2]).all()
assert places_registry["place_fips"].is_unique
print("OK registry:", places_registry.groupby("tier").size().to_dict())
CO_COUNTIES = sorted({c for cs in places_registry["county_fips_json"].map(json.loads) for c in cs if c.startswith("08")})
ALL_COUNTIES = sorted({c for cs in places_registry["county_fips_json"].map(json.loads) for c in cs})
PLACE_FIPS = places_registry["place_fips"].tolist()
print("CO counties:", CO_COUNTIES, "| all counties:", len(ALL_COUNTIES))

  registry live failed: fetch() missing 1 required positional argument: 'fname'
  [synthetic] places_registry            rows=6        -> data/processed/places_registry.csv
OK registry: {1: 6}
CO counties: ['08013', '08123'] | all counties: 2


## 3 · Single-year-of-age  ·  *live: DOLA (Colorado); scaffold+fallback: NHGIS (places & non-CO counties)*

DOLA publishes **real single-year-of-age 1990–2060 for every Colorado county** — this directly
serves the CO-county rebuild base, the CO control trajectory, and the Colorado reconciliation
layer. Place-level single-year age (the H1 outcome) and non-CO peer counties require an **NHGIS**
extract (`ipumspy`, needs an API key); the real extract code is wired below and falls back to a
synthetic stand-in when no key is present.

In [5]:
# --- DOLA real: Colorado county single-year-of-age (countyfips is county portion only) ---
# dola_mode = "synthetic"; dola_src = "synthetic"
dola_df = None
if LIVE:
    try:
        p = fetch("https://storage.googleapis.com/co-publicdata/sya-county.csv",
                  "dola_sya_county.csv")
        raw = pd.read_csv(p, skiprows=1, dtype=str)
        raw.columns = [c.strip().lower() for c in raw.columns]
        raw["fips"] = "08" + raw["countyfips"].str.zfill(3)
        raw["year"] = raw["year"].astype(int)
        raw["age"] = raw["age"].astype(int)
        raw["pop"] = pd.to_numeric(raw["totalpopulation"], errors="coerce")
        dola_df = cap_ages_85(raw[["fips", "year", "age", "pop"]], keys=("fips", "year"))
        dola_mode = "live"; dola_src = "googleapis:co-publicdata/sya-county.csv (DOLA V2024)"
    except Exception as ex:
        print("  DOLA live failed:", ex)
        if not ALLOW_FALLBACK: raise
if dola_df is None:
    dola_df = synth_age_long(CO_COUNTIES, list(range(1990, 2061)), "fips", 10)
# reconciliation file (CO single-year, all years) + the CO slice of county history
dola_recon = dola_df[dola_df["fips"].isin(CO_COUNTIES)].copy()
dola_recon.to_csv(PROC / "dola_reconciliation_syoa.csv", index=False)
record("dola_reconciliation_syoa", PROC / "dola_reconciliation_syoa.csv", dola_mode, dola_src, len(dola_recon))
assert dola_recon["age"].between(0, MAX_AGE).all()
assert (dola_recon["pop"] >= 0).all()
print("OK DOLA:", dola_recon["fips"].nunique(), "CO counties, years",
      dola_recon["year"].min(), "-", dola_recon["year"].max())

  [live     ] dola_reconciliation_syoa   rows=12212    -> data/processed/dola_reconciliation_syoa.csv
OK DOLA: 2 CO counties, years 1990 - 2060


In [6]:
# --- NHGIS place + non-CO county single-year 1990-2020 (real path scaffolded) ---
def nhgis_extract_syoa(geog, fips_list):
    # REAL PATH (requires IPUMS_API_KEY): build+submit+download an NHGIS single-year-of-age
    # extract via ipumspy, then reshape to long [fips, year, age, pop]. Returns df or None.
    key = os.environ.get("IPUMS_API_KEY")
    if not key:
        return None
    try:
        from ipumspy import IpumsApiClient, AggregateDataExtract  # noqa
        # client = IpumsApiClient(key)
        # ext = AggregateDataExtract(collection="nhgis", datasets=[...age-by-sex SYOA...])
        # client.submit_extract(ext); client.wait_for_extract(ext); client.download_extract(ext, RAW)
        # ... read the shipped CSV(s), pivot sex+age to single-year totals, melt to long ...
        raise NotImplementedError("Wire ipumspy datasets/tables, then remove this raise.")
    except Exception as ex:
        print(f"  NHGIS {geog} live unavailable ({type(ex).__name__}); using fallback")
        return None

# places: try NHGIS, else synthetic; CO counties: from DOLA (real); non-CO counties: NHGIS/synth
place_age = nhgis_extract_syoa("place", PLACE_FIPS)
place_mode = "live" if place_age is not None else "synthetic"
if place_age is None:
    place_age = synth_age_long(PLACE_FIPS, BASE_YEARS, "place_fips", 1)
place_age = place_age[place_age["year"].isin(BASE_YEARS)]
place_age.to_csv(PROC / "place_age_syoa.csv", index=False)
record("place_age_syoa", PROC / "place_age_syoa.csv", place_mode,
       "nhgis (or synthetic fallback)", len(place_age))

co_hist = dola_df[dola_df["fips"].isin(CO_COUNTIES) & dola_df["year"].isin(BASE_YEARS)].rename(columns={"fips": "county_fips"})
non_co = [c for c in ALL_COUNTIES if c not in CO_COUNTIES]
nhgis_cty = nhgis_extract_syoa("county", non_co)
cty_mode = "live" if (nhgis_cty is not None and dola_mode == "live") else ("mixed" if dola_mode == "live" else "synthetic")
if nhgis_cty is None:
    nhgis_cty = synth_age_long(non_co, BASE_YEARS, "county_fips", 5)
county_age = pd.concat([co_hist[["county_fips", "year", "age", "pop"]], nhgis_cty], ignore_index=True)
county_age.to_csv(PROC / "county_age_syoa.csv", index=False)
record("county_age_syoa", PROC / "county_age_syoa.csv", cty_mode,
       "DOLA (CO) + nhgis/synthetic (non-CO)", len(county_age))
# validate both
for nm, d, idc in [("place_age", place_age, "place_fips"), ("county_age", county_age, "county_fips")]:
    assert set(d["year"]).issubset(set(BASE_YEARS))
    assert d["age"].between(0, MAX_AGE).all() and (d["pop"] >= 0).all()
    assert d.groupby([idc, "year"]).size().eq(MAX_AGE + 1).all(), f"{nm}: not 0..85 per group"
print("OK SYOA: place mode", place_mode, "| county mode", cty_mode)

  [synthetic] place_age_syoa             rows=2064     -> data/processed/place_age_syoa.csv
  [mixed    ] county_age_syoa            rows=688      -> data/processed/county_age_syoa.csv
OK SYOA: place mode synthetic | county mode mixed


## 4 · Migration flows  ·  *live: IRS county→county (1990–2010)*

The published Hauer flat file is real and county-to-county; the within-county diagonal (non-movers)
and the masked `99999` (<10 filers) are dropped. **Vintage ends 2010** — extension to ~2022 via the
IRS SOI annual files (and the ~2011 format seam) is a documented TODO, not done here.

In [ ]:
irs_mode = "synthetic"; irs_src = "synthetic"; irs_long = None
if LIVE:
    try:
        p = fetch("https://raw.githubusercontent.com/mathewhauer/IRS-migration-data/"
                  "master/DATA-PROCESSED/county_migration_data.txt", "irs_county_migration.txt")
        wide = pd.read_csv(p, sep="\t", dtype={"origin": str, "destination": str})
        yr_cols = [c for c in wide.columns if c.strip().isdigit()]
        long = wide.melt(id_vars=["origin", "destination"], value_vars=yr_cols,
                         var_name="year", value_name="n_migrants")
        long["origin_fips"] = long["origin"].str.zfill(5)
        long["dest_fips"] = long["destination"].str.zfill(5)
        long["year"] = long["year"].astype(int)
        long["n_migrants"] = pd.to_numeric(long["n_migrants"], errors="coerce").fillna(0).astype(int)
        long = long[(long["origin_fips"] != long["dest_fips"]) &        # drop non-movers
                    (long["dest_fips"] != "99999") & (long["origin_fips"] != "99999")]
        irs_long = long[["origin_fips", "dest_fips", "year", "n_migrants"]]
        irs_mode = "live"; irs_src = "github:mathewhauer/IRS-migration-data (1990-2010)"
    except Exception as ex:
        print("  IRS live failed:", ex)
        if not ALLOW_FALLBACK: raise
if irs_long is None:
    pool = ALL_COUNTIES + ["08123", "08069", "08001", "08005"]
    rr = [(o, str(d), y, int(RNG.uniform(20, 400)))
          for o in ALL_COUNTIES for y in range(2000, 2011)
          for d in RNG.choice(pool, 5, replace=False) if d != o]
    irs_long = pd.DataFrame(rr, columns=["origin_fips", "dest_fips", "year", "n_migrants"])
irs_long.to_csv(PROC / "irs_flows.csv", index=False)
record("irs_flows", PROC / "irs_flows.csv", irs_mode, irs_src, len(irs_long))
assert irs_long["origin_fips"].str.len().eq(5).all() and irs_long["dest_fips"].str.len().eq(5).all()
assert (irs_long["n_migrants"] >= 0).all() and (irs_long["origin_fips"] != irs_long["dest_fips"]).all()
print("OK IRS flows:", len(irs_long), "edges, years", irs_long.year.min(), "-", irs_long.year.max())

## 5 · County control (SSP2)  ·  *scaffold + fallback: Hauer county projections*

The uniform nationwide control is Hauer's SSP2 county projection (5-year age groups, to 2050).
The OSF bulk file (`SSP_asrc.csv.zip`, all US counties × age × sex × race × SSP) is large; the
real download/filter is wired below and falls back to a synthetic 5-year control. The **single-year
rebuild** of this control is modeling and lives in notebook 02 (Module 3).

In [ ]:
AGE5 = [(0,4),(5,9),(10,14),(15,19),(20,24),(25,29),(30,34),(35,39),(40,44),
        (45,49),(50,54),(55,59),(60,64),(65,69),(70,74),(75,79),(80,84),(85,200)]
def lbl(a,b): return f"{a}-{b}" if b < 200 else "85+"
ctrl_years = [2020] + PROJ_YEARS
hauer_mode = "synthetic"; hauer_src = "synthetic"; hauer_df = None
if LIVE:
    try:
        # REAL PATH: OSF DOI 10.17605/OSF.IO/9YNFC -> SSP_asrc.csv.zip (large).
        # p = fetch("https://osf.io/download/<file-id>/", "hauer_SSP_asrc.csv.zip")
        # h = pd.read_csv(p, compression="zip", dtype={"GEOID": str})
        # h = h[(h.SCENARIO=="SSP2") & h.GEOID.isin(ALL_COUNTIES) & h.YEAR.isin(ctrl_years)]
        # hauer_df = h.groupby(["GEOID","YEAR","AGEGROUP"], as_index=False)["value"].sum()...
        raise NotImplementedError("Resolve OSF file id; large download guarded off by default.")
    except Exception as ex:
        print("  Hauer OSF not pulled here ({}); using synthetic 5yr control".format(type(ex).__name__))
        if not ALLOW_FALLBACK: raise
if hauer_df is None:
    rng = np.random.default_rng(SEED + 7); recs = []
    for c in ALL_COUNTIES:
        base = rng.uniform(2_000, 60_000, len(AGE5))
        drift = rng.normal(0, 0.05, len(ctrl_years)).cumsum()
        for j, y in enumerate(ctrl_years):
            for (a, b), v in zip(AGE5, base * (1 + drift[j])):
                recs.append((c, y, lbl(a, b), max(v, 0.0)))
    hauer_df = pd.DataFrame(recs, columns=["county_fips", "year", "age_group", "pop"])
hauer_df.to_csv(PROC / "county_control_5yr.csv", index=False)
record("county_control_5yr", PROC / "county_control_5yr.csv", hauer_mode, hauer_src, len(hauer_df))
assert set(hauer_df["year"]) == set(ctrl_years) and (hauer_df["pop"] >= 0).all()
assert hauer_df["age_group"].nunique() == len(AGE5)
print("OK control:", hauer_df["county_fips"].nunique(), "counties x", len(ctrl_years), "years (SSP2, 5yr)")

## 6 · H2 covariates  ·  *live: Zillow ZHVI; scaffold+fallback: BPS permits, NZA restrictiveness*

Zillow city ZHVI is filtered to the registry to derive 2020 level and 2000→2020 growth. Building
permits (Census BPS) and zoning restrictiveness (National Zoning Atlas) are scaffolded with
fallback. NZA is a **current snapshot** vs. 2000–2020 outcomes — a directional bias carried into
notebook 02's limits.

In [ ]:
STATE_ABBR = {"Colorado":"CO","Michigan":"MI","Wisconsin":"WI","California":"CA","Massachusetts":"MA",
              "Utah":"UT","Texas":"TX","Iowa":"IA","North Carolina":"NC","Arizona":"AZ","Oregon":"OR",
              "Florida":"FL","Indiana":"IN"}
# city/state match keys from registry names like "Boulder (Colorado)" or "Boulder city, CO"
def parse_city_state(nm):
    if "(" in nm:
        city = nm.split("(")[0].strip(); st = nm.split("(")[1].rstrip(")").strip()
        return city, STATE_ABBR.get(st, st[:2].upper())
    city = nm.split(" city")[0].split(" town")[0].split(",")[0].strip()
    st = nm.split(",")[-1].strip()
    return city, st
reg_keys = {pf: parse_city_state(nm) for pf, nm in zip(places_registry.place_fips, places_registry.name)}

zhvi_mode = "synthetic"; zhvi = {}
if LIVE:
    try:
        p = fetch("https://files.zillowstatic.com/research/public_csvs/zhvi/"
                  "City_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv", "zillow_zhvi_city.csv", timeout=180)
        z = pd.read_csv(p, dtype={"RegionName": str, "State": str})
        z["key"] = list(zip(z["RegionName"], z["State"]))
        want = set(reg_keys.values())
        zf = z[z["key"].isin(want)].copy()
        c2020 = [c for c in z.columns if c.startswith("2020-")]
        c2000 = [c for c in z.columns if c.startswith("2000-")]
        for pf, k in reg_keys.items():
            row = zf[zf["key"] == k]
            if len(row):
                v20 = row[c2020].mean(axis=1).iloc[0]; v00 = row[c2000].mean(axis=1).iloc[0]
                zhvi[pf] = (float(v20), float(v20 / v00 - 1) if v00 and not np.isnan(v00) else np.nan)
        zhvi_mode = "live" if zhvi else "synthetic"
    except Exception as ex:
        print("  Zillow live failed:", ex)
        if not ALLOW_FALLBACK: raise
# assemble covariates: zhvi (real where matched) + permits/nza (fallback) 
recs = []
for pf in PLACE_FIPS:
    z20, zg = zhvi.get(pf, (np.nan, np.nan))
    if z20 is None or np.isnan(z20):
        z20 = float(RNG.uniform(250_000, 1_400_000))
    if zg is None or np.isnan(zg):           # Zillow may lack 2000 coverage for a matched city
        zg = float(RNG.uniform(0.4, 2.6))
    recs.append((pf, float(RNG.uniform(0.2, 0.95)), z20, zg, float(RNG.uniform(0.5, 12.0))))
cov = pd.DataFrame(recs, columns=["place_fips", "nza_restrictiveness", "zhvi_2020",
                                  "zhvi_growth_00_20", "permits_per_1k"])
cov.to_csv(PROC / "covariates.csv", index=False)
n_real_zhvi = sum(1 for pf in PLACE_FIPS if pf in zhvi)
record("covariates", PROC / "covariates.csv",
       f"zhvi:{zhvi_mode}({n_real_zhvi}/{len(PLACE_FIPS)}); permits:synthetic; nza:synthetic",
       "zillow + bps(scaffold) + nza(scaffold)", len(cov))
assert cov["nza_restrictiveness"].between(0, 1).all() and (cov.drop(columns="place_fips") >= 0).all().all()
print(f"OK covariates: {n_real_zhvi}/{len(PLACE_FIPS)} ZHVI matched live; permits+NZA synthetic")

## 7 · Provenance manifest & contract summary

`manifest.json` records mode/source/rows/sha256 for every processed file. Notebook 02 reads only
`data/processed/` and never re-downloads. The summary makes explicit which inputs are real here
(CO ages, prices, flows, registry) and which are synthetic pending keyed/gated pulls.

In [ ]:
(PROC / "manifest.json").write_text(json.dumps(MANIFEST, indent=2))
man = pd.DataFrame(MANIFEST)[["name", "mode", "rows", "sha256_16", "source_url"]]
print("CONTRACT WRITTEN to data/processed/  (read by 02-boulder-enrollment-demography.ipynb)\n")
print(man.to_string(index=False))
real = man["mode"].str.contains("live").sum()
print(f"\n{real}/{len(man)} outputs include live data. "
      "Set IPUMS_API_KEY (NHGIS) and resolve the Hauer OSF id to lift the rest to live.")